# learning-to-focus on colab

Use an A100 or L4. Mount Drive first so outputs survive the VM getting recycled,
otherwise they land in /content and disappear with it.

Run the setup cells, then just the experiment you want. Everything resumes, so if the
runtime dies, rerun setup and the same cell again.

In [ ]:
# if this fails, close other google accounts in this browser and try again.
# without drive, outputs go to /content/learning-to-focus/outputs and vanish with the VM
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as e:
    print("drive mount failed, writing to local disk instead:", e)

In [ ]:
%cd /content
!test -d learning-to-focus && (cd learning-to-focus && git pull -q) || git clone -q https://github.com/Adi-Kaul/learning-to-focus.git
%cd /content/learning-to-focus
!git log --oneline | head -3

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# optional, skip if you don't use wandb
import wandb
wandb.login()

In [ ]:
# split is deterministic, the diff checks it against the committed test ids
!python -m src.data --test-per-bucket 20
!diff <(python -c "import json;[print(json.loads(l)['id']) for l in open('data/niah_test.jsonl')]") data/splits/niah_test_ids.txt && echo "test split matches"
!python -m src.synth

In [ ]:
!python -m pytest -q tests/

## smoke test

In [ ]:
!python -m src.generate --config configs/smoke_niah.yaml

## warm-up on fake documents (0.6B)
Before / SFT / after on the synthetic set. Mostly to see the training loop work end to end.

In [ ]:
!python -m src.generate --model Qwen/Qwen3-0.6B --data data/synth_test.jsonl --arms da_nomask da --max-new-tokens 256 --batch-size 1 --out-dir '$$LTF_ROOT/runs' --run-name synth_0.6b_before
!python -m src.train_sft --config configs/sft_synth_0.6b.yaml
!python -m src.generate --model Qwen/Qwen3-0.6B --adapter '$$LTF_ROOT/checkpoints/sft_synth_0.6b/final' --data data/synth_test.jsonl --arms da_nomask da --max-new-tokens 256 --batch-size 1 --out-dir '$$LTF_ROOT/runs' --run-name synth_0.6b_after

## baseline: vanilla vs DA without mask vs DA
Do `--buckets 4k` first to see if the 4B falls apart under the mask at all.

In [ ]:
!python -m src.generate --config configs/baseline_qwen3_4b.yaml --buckets 4k --run-name baseline_qwen3_4b_4k
# !python -m src.generate --config configs/baseline_qwen3_4b.yaml

## SFT on the model's own good traces
Sample under the mask and keep the traces that got it right. Try 50 questions first and check how
many survive. If almost none do, `--ideal` writes templated traces instead.

In [ ]:
!python -m src.sample_traces --model Qwen/Qwen3-4B --n-questions 50 --k 8 --buckets 4k 8k --out sft_kept.jsonl

In [ ]:
!python -m src.train_sft --config configs/sft_qwen3_4b.yaml
!python -m src.generate --config configs/eval_sft_qwen3_4b.yaml --run-name eval_sft_qwen3_4b

## GRPO
Vanilla reference costs for the reward first, then a short pilot. Read the logged completions
for reward hacking before doing the real run.

In [ ]:
!python -m src.sample_traces --vanilla-ref --model Qwen/Qwen3-4B --n-questions 400 --buckets 4k --out grpo_train.jsonl

In [ ]:
!python -m src.train_grpo --config configs/grpo_qwen3_4b.yaml --max-steps 20 --out-dir '$$LTF_ROOT/checkpoints/grpo_pilot'

In [ ]:
!python -m src.train_grpo --config configs/grpo_qwen3_4b.yaml
!python -m src.generate --config configs/eval_grpo_qwen3_4b.yaml --run-name eval_grpo_qwen3_4b

## download the small stuff
CSVs and meta only, traces are too big. These go in `results/` in the repo.

In [ ]:
import os, sys; sys.path.insert(0, '.')
from src.paths import root
os.chdir(root() + '/runs')
!zip -qr /content/results_small.zip */summary.csv */metrics.csv */meta.json && ls -la /content/results_small.zip
os.chdir('/content/learning-to-focus')
from google.colab import files
files.download('/content/results_small.zip')